# Analyse Technique BVC - Notebook d'exemple

Ce notebook montre comment utiliser les outils d'analyse technique pour les actions de la Bourse de Casablanca.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.data import BVCDataFetcher
from src.data.tickers import BVC_TICKERS, list_sectors
from src.analysis import TechnicalAnalyzer
from src.visualization.charts import plot_chart, plot_indicators
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. Récupération des données

In [ ]:
fetcher = BVCDataFetcher()

# Récupérer les données d'Attijariwafa Bank sur 1 an
df = fetcher.get_ohlcv('ATW', period='1y')
print(f"Données récupérées: {len(df)} jours")
df.tail(5)

## 2. Rapport d'analyse complet

In [ ]:
analyzer = TechnicalAnalyzer(df)
print(analyzer.full_report())

## 3. Graphique complet avec chandeliers

In [ ]:
fig = plot_chart(
    analyzer.compute_all(),
    show_volume=True,
    show_bollinger=True,
    show_macd=True,
    show_rsi=True,
    figsize=(16, 12)
)
plt.show()

## 4. Signaux et score

In [ ]:
signals = analyzer.get_signals()
for name, sig in signals.items():
    print(f"{name:<20}: {sig['signal']:<12} | {sig.get('note', '')}")

print()
score = analyzer.score()
print(f"Score global: {score['score']:+.1f}/100")
print(f"Recommandation: {score['recommandation']}")

## 5. Supports et résistances

In [ ]:
sr = analyzer.support_resistance()
print(f"Prix actuel: {sr['prix_actuel']} MAD")
print(f"Résistances: {sr['resistances']}")
print(f"Supports: {sr['supports']}")

## 6. Aperçu du marché BVC

In [ ]:
overview = fetcher.get_market_overview(period='1mo')
overview.style.background_gradient(subset=['Variation (%)'], cmap='RdYlGn', vmin=-5, vmax=5)

## 7. Comparer plusieurs actions

In [ ]:
symboles = ['ATW', 'BCP', 'BOA', 'CIH']
resultats = []

for sym in symboles:
    df_sym = fetcher.get_ohlcv(sym, period='6mo')
    if not df_sym.empty:
        an = TechnicalAnalyzer(df_sym)
        sc = an.score()
        summ = an.summary()
        resultats.append({
            'Symbole': sym,
            'Cours': summ['dernier_cours'],
            'Var 1j (%)': summ['variation_1j'],
            'Var 1m (%)': summ['variation_1m'],
            'Score AT': sc['score'],
            'Signal': sc['recommandation']
        })

import pandas as pd
pd.DataFrame(resultats).set_index('Symbole')

## 8. Indicateurs spécifiques

In [ ]:
# Afficher RSI, MACD et Stochastique
fig = plot_indicators(
    analyzer.compute_all(),
    indicators=['RSI', 'MACD', 'Stochastique'],
    figsize=(16, 10)
)
plt.show()